- https://verl.readthedocs.io/en/latest/advance/fully_async.html
    - why fully async
- full async 基本等同于 recipe/fully_async_policy 这套训练框架

### core codes

> verl/examples/run_mimo_7b_mtp_fully_async_megatron_multinode.sh

- verl/verl/experimental/fully_async_policy/shell/
    - `verl.experimental.fully_async_policy.fully_async_main`

```python
# fully_async_main.py
rollouter_future = self.components["rollouter"].fit.remote()
trainer_future = self.components["trainer"].fit.remote()
```
- 两边通过 Ray actor MessageQueue 解耦。rollouter 生成完一个 sample 后 put_sample。trainer 则循环 get_sample，直到收够 required_samples。
```python
# fully_async_rollouter.py
ret = await self.async_rollout_manager.generate_sequences_single(...)
...
success = await self.message_queue_client.put_sample(...)

# fully_async_trainer.py
while len(queue_samples) < self.required_samples:
    sample, queue_len = await self.message_queue_client.get_sample()
```

### basics

> verl 的 fully async 是“RL trainer/rollouter 解耦 + queue + freshness/version 控制”。rollout producer and trainer consumer overlap; MessageQueue + policy version freshness replace the synchronous batch barrier

- RL 训练管线从：
    - $\theta_k \rightarrow \text{rollout 一整批} \rightarrow \text{trainer 更新} \rightarrow \theta_{k+1}$
- 变成：
    - $\text{rollouter 持续生产样本} \rightarrow \text{MessageQueue} \rightarrow \text{trainer 持续消费并更新}$
- fully async 改的是“数据怎么来”和“多久同步 rollout 参数”，即fully async 改的是 rollout 和 trainer 的生产/消费关系，不是把 PPO 内部的 mini-batch / micro-batch 训练逻辑取消掉。
- 训练流程
    - rollouter 侧每次从 dataloader 取一个样本，先按 rollout.n repeat 成多条 response：
    - 生成完成后，一个 RolloutSample 放进 MessageQueue。trainer 不再自己同步 rollout，而是在 _get_samples_from_queue() 里攒够：
        - $\text{required\_samples}=\text{ppo\_mini\_batch\_size}\times\text{require\_batches}$
            - require_batches: prompt-level samples
        - $M=\text{actor.ppo\_mini\_batch\_size},\quad R=\text{async\_training.require\_batches},\quad n=\text{rollout.n}$
            - $E =$ actor.ppo_epochs
            - 默认 fully async config 里 require_batches: 1
        - 那么 trainer 一次 fit_step 收到的是 \(RM\) 个 prompt/sample，但真正训练的 trajectory 数是：$B_{\text{traj}} = R M n$
    - mini update / micro update
        - actor 更新时，`_update_actor()` 会把 PPO mini-batch size 设成：$M_{\text{traj}} = M \times n$
        - 然后 worker 的 train_mini_batch() 会按这个 mini_batch_size 和 ppo_epochs 做 dataloader 迭代，
        - 所以一次 fully async fit_step 的 actor optimizer step 数大致是：$\#\text{actor updates per fit\_step}=R \times \text{ppo\_epochs}$
        - $\text{zero\_grad}\rightarrow\text{多个 micro-batch forward/backward}\rightarrow\text{optimizer\_step}$
- weight sync
    - 一次 outer step 会从队列收 $M R$ 个 prompt-level rollout samples；展开 responses 后大约是 $M R n$ 条 trajectory。actor update 里又把 PPO mini-batch size 设成 $M n$：
    - 因此一次 outer step 内部通常会有 $R \times E$ 个 actor mini-batch optimizer updates。真正 optimizer.step() 在 engine 的 train_batch() 里：
    - 所以 async_training.trigger_parameter_sync_step=4 的精确含义是：
        - 每完成 4 次 FullyAsyncTrainer.fit_step() 后，同步一次 actor weights 给 rollout engine。
        - $\text{actor optimizer steps between rollout syncs}\approx 4 \times R \times E$

### bypass vs. decoupled

```
# fully_async_ppo_trainer.yaml
rollout_is: null
rollout_rs: null
loss_type: ppo_clip
```
- roll_corr
    - token/sequence IS + TIS/IcePop + RS
        - RS 是否是 seq 级别  

$$
\rho_t(\theta)=\exp(\log \pi_\theta(a_t|h_t)-\log \pi_{\text{old}}(a_t|h_t))
$$

> fully async 后，样本可能来自更早的 rollout policy：$\mu_i = \pi_{\text{rollout version } i}$。所以正确的分母应该变成 $\mu_i$，也就是该样本真正生成时的策略，而不是当前 trainer 随便 recompute 出来的 policy。

- bypass
    - $\text{old\_log\_probs} := \text{rollout\_log\_probs}$
    - PPO loss 里的 ratio 变成：$\rho_t(\theta)=\exp(\log \pi_\theta(a_t|h_t)-\log \pi_{\text{rollout}}(a_t|h_t))$
        - rollout_corr_helper.py: apply_bypass_mode
- decoupled：即 bypass_mode=False，verl 进入三策略形式：$\pi_{\text{rollout}} \rightarrow \pi_{\text{old}} \rightarrow \pi_\theta$
    - 其中 PPO ratio 用：$r_t = \frac{\pi_\theta(a_t|h_t)}{\pi_{\text{old}}(a_t|h_t)}$
    - rollout correction 再提供：$w_t = \frac{\pi_{\text{old}}(a_t|h_t)}{\pi_{\text{rollout}}(a_t|h_t)}$
    - 两者相乘，在未 clip / 未 truncate 的理想情况下就是：$r_t w_t =\frac{\pi_\theta(a_t|h_t)}{\pi_{\text{rollout}}(a_t|h_t)}$
- 三策略的内涵
    - $\pi_\theta / \pi_{\text{old}}$：PPO 的 proximal update ratio，表示“当前 actor 相对本轮训练开始时的 actor 移动了多少”。这是可微的、进 PPO clip 的部分。
    - $\pi_{\text{old}} / \pi_{\text{rollout}}$：rollout correction ratio，表示“这批数据的行为策略和训练侧旧策略有多不一致”。这是采样分布校正，通常 detach、可 truncation、可 rejection sampling。
    - PPO clipping 本来应该限制“本次 optimizer update 让 policy 移动多少”，而不是惩罚“rollout worker 因为异步而落后了多少”。
        - 举个直觉例子：假设 rollout policy 很旧，$\pi_{\text{rollout}}$ 已经和 trainer 当前旧参数 $\pi_{\text{old}}$ 差很多。训练刚开始时 $\pi_\theta=\pi_{\text{old}}$。如果你直接用：$r = \frac{\pi_\theta}{\pi_{\text{rollout}}}$那么一开始 $r$ 就可能远离 1，PPO clip 会认为“当前 policy 已经偏离太多”，但这不是这一步 update 造成的，而是异步系统的历史 lag 造成的。这样 clip 的语义就混了。
        - decoupled mode 下，一开始：$\frac{\pi_\theta}{\pi_{\text{old}}}=1$
        - 所以 PPO clip 仍然只约束 trainer 这轮更新的局部步长；而旧 rollout 带来的 off-policy 问题由：$\frac{\pi_{\text{old}}}{\pi_{\text{rollout}}}$ 单独处理。

### 调度层 mitigation

> staleness scheduling

- staleness_threshold 限制最多允许多少 stale samples。
- trigger_parameter_sync_step 控制 trainer 做几次本地更新后同步 rollout 参数。
- MessageQueue 解耦生产/消费，但 rollouter 会在 queue 或 staleness 超限时 pause。
- partial_rollout 处理长尾生成，减少参数同步时等待 in-flight rollout 的时间。

这些不是 rollout correction 的数学 IS/RS 机制，而是 fully async 的系统层 freshness control。

### token-level vs. seq-level

- rollout_correction 里的 IS 理解成：样本是由 rollout policy $\mu$ 生成的，但训练想更新某个 training policy $\pi$，所以要用概率比把分布从 $\mu$ 换到 $\pi$。
    - $\Delta_t = \log \pi(y_t \mid h_t) - \log \mu(y_t \mid h_t)$
    - `log_ratio = old_log_prob - rollout_log_prob`
    - 这里的 old_log_prob 在 decoupled mode 里是 trainer 侧 recompute 的 old policy；在 bypass mode 里可以是 current policy logprob。rollout_log_prob 是样本实际生成时 rollout engine 记录下来的行为策略 logprob。
- Token-Level IS：token-level 就是每个 token 自己算一个权重
    - $w_t^{\text{token}} = \exp(\Delta_t)= \frac{\pi(y_t \mid h_t)}{\mu(y_t \mid h_t)}$
$$L_{\text{token-IS}}=-\sum_t w_t^{\text{token}} A_t \log \pi_\theta(y_t \mid h_t)$$
```python
log_ratio_safe = torch.clamp(log_ratio, min=-SAFETY_BOUND, max=SAFETY_BOUND)
raw_rollout_is_weights = torch.exp(log_ratio_safe)
```
- Sequence-Level IS：sequence-level 是先把整条 response 的 log-ratio 加起来：$\Delta_{\text{seq}}=\sum_{t \in \text{valid}} \Delta_t$
    - 然后得到整条序列的权重：$w^{\text{seq}}=\exp(\Delta_{\text{seq}})=\prod_t\frac{\pi(y_t \mid h_t)}{\mu(y_t \mid h_t)}$
```python
log_ratio_sum = masked_sum(log_ratio, response_mask, axis=-1).unsqueeze(-1)
raw_rollout_is_weights = torch.exp(log_ratio_sum_safe).expand_as(log_ratio)
```

$$
L_{\text{seq-IS}}=-\sum_t w^{\text{seq}} A_t \log \pi_\theta(y_t \mid h_t)
$$

------

如果 rollout_is_threshold 写成 "lower_upper"，就是 IcePop：只保留区间内的权重，区间外置 0：
$$
\tilde w =
\begin{cases}
w, & L \le w \le U \\
0, & \text{otherwise}
\end{cases}
$$

### misc

- RS 和 IS weight 是两条机制。IS 是“乘权重”；RS 是“改 mask”。
    - RS 会根据 token_k1/k2/k3 或 seq_sum/seq_mean/seq_max 这些 divergence statistic 判断是否超出 hard trust region。最后返回 modified_response_mask。
-  bypass + loss_type="ppo_clip" 时，verl 不会再把 rollout_is_weights 乘到 loss 上，因为 PPO ratio 已经是：
    -  $r_t =\frac{\pi_\theta(y_t \mid h_t)}{\pi_{\text{rollout}}(y_t \mid h_t)}$
    -  再乘 IS 会 double count。
- 显式 token/seq IS 权重主要在 decoupled mode，或者 bypass + loss_type="reinforce" 时真的乘进 loss；PPO-clip 路径里更多是利用 ratio clipping 和 RS 来控制 off-policy gap。 